<a href="https://colab.research.google.com/github/sunilllinus/CYO/blob/master/HW3_Colab_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW3 Unit Conversion — Full Colab Training Pipeline

**Purpose**: Train SFT and RFT models on Google Colab (T4/A100) faster and better than local Mac.

## Key improvements over Mac baseline
| What | Mac (baseline) | Colab (this notebook) |
|---|---|---|
| SFT accuracy | 0.51 | Target ≥ 0.60 |
| RFT accuracy | not yet run | Target ≥ 0.70 (bonus ≥ 0.80) |
| LoRA rank | r=8 | r=16 (more capacity) |
| Epochs | 5 | 10 |
| Training format | raw question | chat template (matches base model priors) |
| RFT max_length | 128 (truncated!) | 512 (full reasoning chains) |
| dtype | float32 on MPS | float16 on CUDA (2× faster) |
| Batch | 8 | 16 (SFT) / 8 (RFT) |

## Steps
1. ✅ GPU check + install packages
2. ✅ Upload data files (train.json, valid.json)
3. 🏋️ SFT Training → target 0.60+
4. 🔄 Datagen: 1.7B model generates reasoning chains → data/rft.json
5. 🏋️ RFT Training → target 0.70+ (bonus: 0.80+)
6. 📦 Download adapters → copy to Mac's homework/sft_model + homework/rft_model

## GPU requirements
- **Free T4 (15 GB VRAM)**: Runs full pipeline sequentially (~2.5 hrs)
- **Pro A100 (40 GB VRAM)**: Larger batches, faster (~1 hr)

> **Before running**: Runtime → Change runtime type → GPU (T4 or A100)

In [ ]:
# ── Cell 1: GPU check ───────────────────────────────────────────────────────
import torch, subprocess

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
    DEVICE = "cuda"
else:
    print("⚠️  No GPU found! Go to Runtime → Change runtime type → GPU")
    DEVICE = "cpu"

try:
    out = subprocess.run(["free", "-h"], capture_output=True, text=True)
    print("\nRAM:\n", out.stdout)
except Exception:
    import psutil
    print(f"\nRAM: {psutil.virtual_memory().total / 1e9:.1f} GB total")

print(f"\nUsing device: {DEVICE}")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB

RAM:
                total        used        free      shared  buff/cache   available
Mem:            12Gi       1.3Gi       7.0Gi       2.0Mi       4.4Gi        11Gi
Swap:             0B          0B          0B


Using device: cuda


In [ ]:
# ── Cell 2: Install packages ─────────────────────────────────────────────────
# Pin transformers to same version as Mac to avoid compatibility surprises.
# peft and accelerate: latest compatible.
!pip install -q "transformers==4.52.4" "peft>=0.15" accelerate tqdm fire

import importlib
for pkg in ["transformers", "peft", "accelerate"]:
    mod = importlib.import_module(pkg)
    print(f"{pkg}: {mod.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 78.8 MB/s eta 0:00:00
transformers: 4.52.4
peft: 0.18.1
accelerate: 1.13.0


In [ ]:
# ── Cell 3: Upload data files ────────────────────────────────────────────────
# Upload train.json and valid.json from your local homework3_v3/data/ folder.
from google.colab import files
import json, os

print("Please upload train.json and valid.json from your Mac's data/ folder...")
uploaded = files.upload()

for fname, content in uploaded.items():
    with open(fname, "wb") as f:
        f.write(content)
    print(f"✅ Saved {fname} ({len(content):,} bytes)")

# Verify
with open("train.json") as f:
    train_data = json.load(f)
with open("valid.json") as f:
    valid_data = json.load(f)

print(f"\nTrain: {len(train_data)} examples")
print(f"Valid: {len(valid_data)} examples")
print(f"Sample: {train_data[0]}")

Please upload train.json and valid.json from your Mac's data/ folder...


Saving train.json to train.json
Saving valid.json to valid.json
✅ Saved train.json (60,782 bytes)
✅ Saved valid.json (61,065 bytes)

Train: 1000 examples
Valid: 1000 examples
Sample: ['Can you change 2 hour to its equivalent in min?', 120.0]


In [ ]:
# ── Cell 4: Shared utilities (tokenize, datasets, helpers) ───────────────────
import torch, json, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel

CHECKPOINT = "HuggingFaceTB/SmolLM2-360M-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def parse_answer(text: str) -> float:
    try:
        return float(text.split("<answer>")[1].split("</answer>")[0])
    except Exception:
        return float("nan")


def is_valid(pred: float, true: float, tol: float = 0.05) -> bool:
    if pred != pred:  # nan
        return False
    return abs(round(pred, 3) - round(true, 3)) < tol * abs(round(true, 3))


def tokenize(tokenizer, question: str, answer: str, max_length: int = 256):
    """Tokenize question+answer, masking question tokens from loss."""
    full_text = f"{question}{answer}{tokenizer.eos_token}"
    tokenizer.padding_side = "right"
    tokenizer.pad_token = tokenizer.eos_token
    full = tokenizer(full_text, padding="max_length", truncation=True, max_length=max_length)
    input_ids = full["input_ids"]
    question_len = len(tokenizer(question)["input_ids"])
    labels = [-100] * question_len + input_ids[question_len:]
    for i in range(len(labels)):
        if full["attention_mask"][i] == 0:
            labels[i] = -100
    full["labels"] = labels
    return full


class TokenizedDataset:
    def __init__(self, tokenizer, data, format_fn, max_length=256):
        self.tokenizer = tokenizer
        self.data = data
        self.format_fn = format_fn
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        fmt = self.format_fn(*self.data[idx])
        return tokenize(self.tokenizer, max_length=self.max_length, **fmt)


def evaluate(model, tokenizer, dataset, n=100, max_new_tokens=200, batch_size=16):
    """Batched evaluation on first n examples of dataset."""
    model.eval()
    tokenizer.padding_side = "left"
    tokenizer.pad_token = tokenizer.eos_token
    n = min(n, len(dataset))
    preds = []
    for i in range(0, n, batch_size):
        batch = [dataset[j][0] for j in range(i, min(i + batch_size, n))]
        inp = tokenizer(batch, return_tensors="pt", padding=True,
                        truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=max_new_tokens,
                                 do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
        il = inp["input_ids"].shape[1]
        for row in out:
            preds.append(parse_answer(tokenizer.decode(row[il:], skip_special_tokens=True)))
    true_answers = [dataset[j][1] for j in range(n)]
    acc = sum(is_valid(p, t) for p, t in zip(preds, true_answers)) / n
    ans_rate = sum(p == p for p in preds) / n
    return acc, ans_rate


print("✅ Utilities defined")

✅ Utilities defined


---
## Part 3 — SFT Training
**Target**: accuracy ≥ 0.60 → 25/25 pts  
**Grading**: score = (acc − 0.40) / 0.20, so every 0.01 accuracy = 1.25 pts  
**Key improvements**: chat template format, r=16, 10 epochs, fp16, max_length=256

In [ ]:
# ── Cell 5: SFT Training ─────────────────────────────────────────────────────
import os
SFT_DIR = "/content/sft_model"
os.makedirs(SFT_DIR, exist_ok=True)

# Format function: use chat template so training format matches base model priors.
# The base model is instruction-tuned; using raw questions misaligns training/inference.
def make_sft_format_fn(tokenizer):
    def fmt(prompt, answer):
        rounded = round(float(answer), 4)
        answer_str = f"<answer>{rounded:g}</answer>"
        messages = [
            {"role": "system", "content": "You are a unit conversion expert. Given a unit conversion question, provide the exact numerical answer inside <answer></answer> tags."},
            {"role": "user", "content": "How many seconds are in 3 minutes?"},
            {"role": "assistant", "content": "<answer>180</answer>"},
            {"role": "user", "content": prompt},
        ]
        q = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        return {"question": q, "answer": answer_str}
    return fmt


print("Loading SmolLM2-360M-Instruct...")
tok_sft = AutoTokenizer.from_pretrained(CHECKPOINT)
mdl_sft = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT, torch_dtype=torch.float16).to(DEVICE)

# r=16 vs r=8 Mac baseline — more LoRA parameters for better expressiveness.
# r=16 adds 8.7M params → total 374.7M < 380M grader limit ✅
lora_sft = LoraConfig(r=16, lora_alpha=32, target_modules="all-linear",
                      bias="none", task_type="CAUSAL_LM")
mdl_sft = get_peft_model(mdl_sft, lora_sft)
mdl_sft.enable_input_require_grads()
mdl_sft.print_trainable_parameters()

sft_dataset = TokenizedDataset(tok_sft, train_data, make_sft_format_fn(tok_sft), max_length=256)
print(f"Training on {len(sft_dataset)} examples, max_length=256")

args_sft = TrainingArguments(
    output_dir=SFT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,     # effective batch = 32
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=20,
    remove_unused_columns=False,
    fp16=True,
    report_to="none",
    dataloader_num_workers=2,
)

trainer_sft = Trainer(model=mdl_sft, args=args_sft, train_dataset=sft_dataset)
print("\n🏋️ Starting SFT training (10 epochs, r=16, fp16, effective batch=32)...")
trainer_sft.train()
trainer_sft.save_model(SFT_DIR)
print(f"\n✅ SFT model saved to {SFT_DIR}")

Loading SmolLM2-360M-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


trainable params: 8,683,520 || all params: 370,504,640 || trainable%: 2.3437
Training on 1000 examples, max_length=256

🏋️ Starting SFT training (10 epochs, r=16, fp16, effective batch=32)...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
20,0.766800
40,0.581700
60,0.444100
80,0.309700
100,0.279600
120,0.198500
140,0.159500
160,0.125300
180,0.085800
200,0.064100



✅ SFT model saved to /content/sft_model


In [ ]:
# ── Cell 6: Evaluate SFT ─────────────────────────────────────────────────────
# Free training model from GPU, then reload as eval model
del mdl_sft; gc.collect(); torch.cuda.empty_cache()

# SFT inference uses the same chat template as training
def make_sft_eval_prompts(tokenizer, dataset, n=100):
    prompts = []
    for i in range(min(n, len(dataset))):
        q = dataset[i][0]
        messages = [
            {"role": "system", "content": "You are a unit conversion expert. Given a unit conversion question, provide the exact numerical answer inside <answer></answer> tags."},
            {"role": "user", "content": "How many seconds are in 3 minutes?"},
            {"role": "assistant", "content": "<answer>180</answer>"},
            {"role": "user", "content": q},
        ]
        prompts.append(tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False))
    return prompts

eval_tok_sft = AutoTokenizer.from_pretrained(CHECKPOINT)
eval_mdl_sft = AutoModelForCausalLM.from_pretrained(CHECKPOINT).to(DEVICE)
eval_mdl_sft = PeftModel.from_pretrained(eval_mdl_sft, SFT_DIR).to(DEVICE)
eval_mdl_sft.eval()

N = min(100, len(valid_data))
prompts = make_sft_eval_prompts(eval_tok_sft, valid_data, N)
eval_tok_sft.padding_side = "left"
eval_tok_sft.pad_token = eval_tok_sft.eos_token

preds_sft = []
BATCH = 16
for i in range(0, N, BATCH):
    batch = prompts[i:i+BATCH]
    inp = eval_tok_sft(batch, return_tensors="pt", padding=True,
                       truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        out = eval_mdl_sft.generate(**inp, max_new_tokens=50, do_sample=False,
                                    pad_token_id=eval_tok_sft.eos_token_id)
    il = inp["input_ids"].shape[1]
    for row in out:
        preds_sft.append(parse_answer(eval_tok_sft.decode(row[il:], skip_special_tokens=True)))

true_sft = [valid_data[i][1] for i in range(N)]
acc_sft = sum(is_valid(p, t) for p, t in zip(preds_sft, true_sft)) / N
ar_sft  = sum(p == p for p in preds_sft) / N

# Grading formula: score = (acc - 0.40) / 0.20, clipped [0,1], × 25
sft_pts = min(max((acc_sft - 0.40) / 0.20, 0.0), 1.0) * 25

print(f"\n📊 SFT Results (vs Mac baseline 0.51):")
print(f"  Accuracy:    {acc_sft:.3f}  |  Answer rate: {ar_sft:.3f}")
print(f"  Estimated score: {sft_pts:.1f} / 25 pts")
print(f"  {'✅ Full marks!' if acc_sft >= 0.60 else f'⚠️  Need 0.60+. Gap: {0.60-acc_sft:.3f}'}")


📊 SFT Results (vs Mac baseline 0.51):
  Accuracy:    0.690  |  Answer rate: 1.000
  Estimated score: 25.0 / 25 pts
  ✅ Full marks!


---
## Part 4 — Datagen (generate RFT reasoning data)
Uses **SmolLM2-1.7B-Instruct** in bfloat16 (~3.4 GB VRAM) to generate 10 CoT candidates per training question.  
Keeps the first candidate whose answer matches. Saves to `rft.json`.

> **IMPORTANT**: Free the SFT eval model first (cell below does this automatically).

In [ ]:
# ── Cell 7: Datagen (1.7B CoT → rft.json) ───────────────────────────────────
# Free SFT eval model first
try:
    del eval_mdl_sft
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()
print(f"GPU free after SFT: {torch.cuda.memory_reserved(0)/1e9:.1f} GB reserved")

DATAGEN_CKPT = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
print(f"\nLoading 1.7B in bfloat16 (~3.4 GB VRAM)...")
tok_dg = AutoTokenizer.from_pretrained(DATAGEN_CKPT)
mdl_dg = AutoModelForCausalLM.from_pretrained(DATAGEN_CKPT, torch_dtype=torch.bfloat16).to(DEVICE)
mdl_dg.eval()
print(f"✅ 1.7B loaded | VRAM used: {torch.cuda.memory_allocated(0)/1e9:.1f} GB")


def cot_prompt(tokenizer, question):
    """Format a question using the same CoT chat template as the training pipeline."""
    messages = [
        {"role": "system", "content": "You are a unit conversion expert. Given a unit conversion question, reason step by step and give the final numerical answer inside <answer></answer> tags. Be concise."},
        {"role": "user", "content": "How many seconds are in 3 minutes?"},
        {"role": "assistant", "content": "1 minute = 60 seconds. 3 * 60 = <answer>180</answer>"},
        {"role": "user", "content": question},
    ]
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)


from tqdm.notebook import tqdm

results = []
OVERSAMPLE = 10
TEMPERATURE = 0.6

print(f"\n🔄 Generating RFT data from {len(train_data)} train examples ({OVERSAMPLE} samples each)...\n")

for item in tqdm(train_data, desc="Datagen"):
    question, correct_answer = item[0], item[1]
    prompt = cot_prompt(tok_dg, question)

    tok_dg.padding_side = "left"
    tok_dg.pad_token = tok_dg.eos_token
    inp = tok_dg([prompt], return_tensors="pt", padding=True,
                  truncation=True, max_length=512).to(DEVICE)
    il = inp["input_ids"].shape[1]

    with torch.no_grad():
        out = mdl_dg.generate(
            **inp,
            max_new_tokens=200,
            do_sample=True,
            temperature=TEMPERATURE,
            num_return_sequences=OVERSAMPLE,
            pad_token_id=tok_dg.eos_token_id,
        )
    generations = [tok_dg.decode(row[il:], skip_special_tokens=True) for row in out]

    for gen in generations:
        if is_valid(parse_answer(gen), correct_answer):
            results.append([question, correct_answer, gen])
            break

with open("rft.json", "w") as f:
    json.dump(results, f, indent=2)

rate = len(results) / len(train_data) * 100
print(f"\n✅ Generated {len(results)}/{len(train_data)} examples ({rate:.1f}% success rate)")
print(f"   Saved to rft.json")

GPU free after SFT: 0.9 GB reserved

Loading 1.7B in bfloat16 (~3.4 GB VRAM)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ 1.7B loaded | VRAM used: 4.3 GB

🔄 Generating RFT data from 1000 train examples (10 samples each)...



Datagen:   0%|          | 0/1000 [00:00<?, ?it/s]


✅ Generated 884/1000 examples (88.4% success rate)
   Saved to rft.json


---
## Part 4 — RFT Training
**Target**: accuracy ≥ 0.70 → 25/25 pts | ≥ 0.80 → 30/25 pts (bonus!)  
**Grading**: base = (acc − 0.60) / 0.10, bonus = (acc − 0.70) / 0.10 × 0.2  
**Critical fix**: max_length=512 (Mac baseline used 128, truncating reasoning chains!)

In [ ]:
# ── Cell 8: RFT Training ─────────────────────────────────────────────────────
# Free 1.7B datagen model
del mdl_dg; gc.collect(); torch.cuda.empty_cache()
print(f"GPU free after datagen: {torch.cuda.memory_reserved(0)/1e9:.1f} GB reserved")

# Load RFT dataset
with open("rft.json") as f:
    rft_data = json.load(f)
print(f"RFT dataset: {len(rft_data)} examples")

RFT_DIR = "/content/rft_model"
os.makedirs(RFT_DIR, exist_ok=True)


def make_rft_format_fn(tokenizer):
    """RFT format: same CoT chat template used during datagen — MUST match inference format."""
    def fmt(question, correct_answer, reasoning):
        messages = [
            {"role": "system", "content": "You are a unit conversion expert. Given a unit conversion question, reason step by step and give the final numerical answer inside <answer></answer> tags. Be concise."},
            {"role": "user", "content": "How many seconds are in 3 minutes?"},
            {"role": "assistant", "content": "1 minute = 60 seconds. 3 * 60 = <answer>180</answer>"},
            {"role": "user", "content": question},
        ]
        q = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        return {"question": q, "answer": reasoning}
    return fmt


print("Loading SmolLM2-360M-Instruct for RFT...")
tok_rft = AutoTokenizer.from_pretrained(CHECKPOINT)
mdl_rft = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT, torch_dtype=torch.float16).to(DEVICE)

lora_rft = LoraConfig(r=16, lora_alpha=64, target_modules="all-linear",
                      bias="none", task_type="CAUSAL_LM")
mdl_rft = get_peft_model(mdl_rft, lora_rft)
mdl_rft.enable_input_require_grads()
mdl_rft.print_trainable_parameters()

# CRITICAL: max_length=512 for RFT!
# Reasoning chains are 100-300 tokens. Mac's max_length=128 was truncating them all,
# which made the model unable to learn complete reasoning → explain the poor baseline.
rft_dataset = TokenizedDataset(tok_rft, rft_data, make_rft_format_fn(tok_rft), max_length=512)
print(f"Training on {len(rft_dataset)} examples, max_length=512")

args_rft = TrainingArguments(
    output_dir=RFT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=8,      # smaller batch for long sequences
    gradient_accumulation_steps=4,      # effective batch = 32
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=20,
    remove_unused_columns=False,
    fp16=True,
    report_to="none",
    dataloader_num_workers=2,
)

trainer_rft = Trainer(model=mdl_rft, args=args_rft, train_dataset=rft_dataset)
print("\n🏋️ Starting RFT training (10 epochs, r=16/alpha=64, fp16, max_length=512)...")
trainer_rft.train()
trainer_rft.save_model(RFT_DIR)
print(f"\n✅ RFT model saved to {RFT_DIR}")

GPU free after datagen: 0.9 GB reserved
RFT dataset: 884 examples
Loading SmolLM2-360M-Instruct for RFT...


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


trainable params: 8,683,520 || all params: 370,504,640 || trainable%: 2.3437
Training on 884 examples, max_length=512

🏋️ Starting RFT training (10 epochs, r=16/alpha=64, fp16, max_length=512)...


Step,Training Loss
20,0.359600
40,0.240300
60,0.201100
80,0.172900
100,0.145000
120,0.133500
140,0.115900
160,0.095200
180,0.083900
200,0.078900



✅ RFT model saved to /content/rft_model


In [ ]:
# ── Cell 9: Evaluate RFT ─────────────────────────────────────────────────────
del mdl_rft; gc.collect(); torch.cuda.empty_cache()

def make_rft_eval_prompts(tokenizer, dataset, n=100):
    prompts = []
    for i in range(min(n, len(dataset))):
        q = dataset[i][0]
        messages = [
            {"role": "system", "content": "You are a unit conversion expert. Given a unit conversion question, reason step by step and give the final numerical answer inside <answer></answer> tags. Be concise."},
            {"role": "user", "content": "How many seconds are in 3 minutes?"},
            {"role": "assistant", "content": "1 minute = 60 seconds. 3 * 60 = <answer>180</answer>"},
            {"role": "user", "content": q},
        ]
        prompts.append(tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False))
    return prompts

eval_tok_rft = AutoTokenizer.from_pretrained(CHECKPOINT)
eval_mdl_rft = AutoModelForCausalLM.from_pretrained(CHECKPOINT).to(DEVICE)
eval_mdl_rft = PeftModel.from_pretrained(eval_mdl_rft, RFT_DIR).to(DEVICE)
eval_mdl_rft.eval()

N = min(100, len(valid_data))
prompts_rft = make_rft_eval_prompts(eval_tok_rft, valid_data, N)
eval_tok_rft.padding_side = "left"
eval_tok_rft.pad_token = eval_tok_rft.eos_token

preds_rft = []
for i in range(0, N, 8):
    batch = prompts_rft[i:i+8]
    inp = eval_tok_rft(batch, return_tensors="pt", padding=True,
                       truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        out = eval_mdl_rft.generate(**inp, max_new_tokens=200, do_sample=False,
                                    pad_token_id=eval_tok_rft.eos_token_id)
    il = inp["input_ids"].shape[1]
    for row in out:
        preds_rft.append(parse_answer(eval_tok_rft.decode(row[il:], skip_special_tokens=True)))

true_rft = [valid_data[i][1] for i in range(N)]
acc_rft = sum(is_valid(p, t) for p, t in zip(preds_rft, true_rft)) / N
ar_rft  = sum(p == p for p in preds_rft) / N

# Grading formula: base (acc-0.60)/0.10 + bonus (acc-0.70)/0.10 * 0.2
import numpy as np
base_pts  = np.clip((acc_rft - 0.60) / 0.10, 0.0, 1.0) * 25
bonus_pts = np.clip((acc_rft - 0.70) / 0.10, 0.0, 1.0) * 5
total_pts = base_pts + bonus_pts

print(f"\n📊 RFT Results:")
print(f"  Accuracy:    {acc_rft:.3f}  |  Answer rate: {ar_rft:.3f}")
print(f"  Base pts: {base_pts:.1f}/25  |  Bonus pts: {bonus_pts:.1f}/5  |  Total: {total_pts:.1f}/30")
if acc_rft >= 0.80:   print("  🎉 FULL BONUS!")
elif acc_rft >= 0.70: print("  ✅ Full marks (25/25). Bonus possible with higher acc.")
elif acc_rft >= 0.60: print(f"  ⚠️  Partial. Gap to full marks (0.70): {0.70-acc_rft:.3f}")
else:                 print(f"  ❌ Below baseline. Gap to 0.60: {0.60-acc_rft:.3f}")

print(f"\n{'='*50}")
print(f"FINAL ESTIMATED SCORE")
print(f"  base_llm/CoT: ~50 pts (batched_generate + CoT)")
print(f"  SFT: {sft_pts:.1f} / 25 pts")
print(f"  RFT: {total_pts:.1f} / 30 pts")
print(f"  Total: ~{50 + sft_pts + total_pts:.0f} / 105 pts")


📊 RFT Results:
  Accuracy:    0.810  |  Answer rate: 0.990
  Base pts: 25.0/25  |  Bonus pts: 5.0/5  |  Total: 30.0/30
  🎉 FULL BONUS!

FINAL ESTIMATED SCORE
  base_llm/CoT: ~50 pts (batched_generate + CoT)
  SFT: 25.0 / 25 pts
  RFT: 30.0 / 30 pts
  Total: ~105 / 105 pts


---
## Download Models

Downloads 3 files:
1. `sft_model_adapter.zip` → unzip → copy contents to `homework/sft_model/` on Mac
2. `rft_model_adapter.zip` → unzip → copy contents to `homework/rft_model/` on Mac  
3. `rft.json` → copy to `data/rft.json` on Mac

Then on Mac, run the grader:
```bash
cd /Users/skp/MyMacDocs/ADL/HW3/homework3_v3
/opt/miniconda3/envs/ADLHW3/bin/python -m grader homework
```

In [ ]:
# ── Cell 10: Package and download ────────────────────────────────────────────
import shutil
from google.colab import files

# Only pack the adapter files (not optimizer state / checkpoints)
# The grader needs: adapter_model.safetensors + adapter_config.json
sft_zip = "/content/sft_model_adapter.zip"
rft_zip = "/content/rft_model_adapter.zip"

shutil.make_archive("/content/sft_model_adapter", "zip", "/content", "sft_model")
shutil.make_archive("/content/rft_model_adapter", "zip", "/content", "rft_model")

import os
print(f"sft_model_adapter.zip: {os.path.getsize(sft_zip)/1e6:.1f} MB")
print(f"rft_model_adapter.zip: {os.path.getsize(rft_zip)/1e6:.1f} MB")
print(f"rft.json:              {os.path.getsize('rft.json')/1e3:.1f} KB")

print("\n📦 Starting downloads...")
files.download(sft_zip)
files.download(rft_zip)
files.download("rft.json")

print("""
✅ Downloads triggered.

Next steps on Mac:
  1. unzip sft_model_adapter.zip
     cp -r sft_model/* /path/to/homework3_v3/homework/sft_model/

  2. unzip rft_model_adapter.zip
     cp -r rft_model/* /path/to/homework3_v3/homework/rft_model/

  3. cp rft.json /path/to/homework3_v3/data/rft.json

  4. Run grader:
     cd /Users/skp/MyMacDocs/ADL/HW3/homework3_v3
     /opt/miniconda3/envs/ADLHW3/bin/python -m grader homework
""")

sft_model_adapter.zip: 128.4 MB
rft_model_adapter.zip: 128.3 MB
rft.json:              135.3 KB

📦 Starting downloads...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Downloads triggered.

Next steps on Mac:
  1. unzip sft_model_adapter.zip
     cp -r sft_model/* /path/to/homework3_v3/homework/sft_model/

  2. unzip rft_model_adapter.zip
     cp -r rft_model/* /path/to/homework3_v3/homework/rft_model/

  3. cp rft.json /path/to/homework3_v3/data/rft.json

  4. Run grader:
     cd /Users/skp/MyMacDocs/ADL/HW3/homework3_v3
     /opt/miniconda3/envs/ADLHW3/bin/python -m grader homework

